In [ ]:
!pip install transformers torch pandas

In [ ]:
import re

In [ ]:
import pandas as pd
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

# Tải mô hình phân tích cảm xúc tiếng Bồ Đào Nha
print("Đang tải mô hình NLP...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="pysentimiento/bertweet-pt-sentiment",
    device=device
)

Đang tải mô hình NLP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Tải thành công!


In [ ]:
# ---------------------------------------------------------
# ĐỊNH NGHĨA TỪ ĐIỂN KHÍA CẠNH (ASPECT DICTIONARY)
# ---------------------------------------------------------
aspect_keywords = {
    # Bỏ 'lindo', 'perfeito', 'quebrado', 'defeito' (để dành cho AI và Override bắt)
    'Product': ['produto', 'qualidade', 'tamanho', 'cor', 'tecido', 'material', 'acabamento', 'costura', 'cheiro'],

    # Bỏ 'chegou', 'demorou' (động từ dễ gây nhiễu)
    'Logistics': ['entrega', 'prazo', 'transportadora', 'caixa', 'correios', 'atraso', 'embalagem', 'rastreio', 'frete'],

    'Seller': ['vendedor', 'loja', 'anúncio', 'nota fiscal', 'falso', 'pirata'],

    'Customer Service': ['reembolso', 'estorno', 'atendimento', 'suporte', 'cancelar', 'devolução']
}

In [ ]:
def analyze_review_aspects_v2(review_text):
    result = {
        'Product_Sentiment': 'NEU',
        'Logistics_Sentiment': 'NEU',
        'Seller_Sentiment': 'NEU',
        'CS_Sentiment': 'NEU'
    }

    if pd.isna(review_text) or str(review_text).strip() == "":
        return result

    review_lower = str(review_text).lower()

    # Bước 1: AI Dự đoán (như cũ)
    has_product = any(word in review_lower for word in aspect_keywords['Product'])
    has_logistics = any(word in review_lower for word in aspect_keywords['Logistics'])
    has_seller = any(word in review_lower for word in aspect_keywords['Seller'])
    has_cs = any(word in review_lower for word in aspect_keywords['Customer Service'])

    if has_product or has_logistics or has_seller or has_cs:
        ai_result = sentiment_analyzer(review_lower[:512])[0]
        sentiment_label = ai_result['label']

        if has_product: result['Product_Sentiment'] = sentiment_label
        if has_logistics: result['Logistics_Sentiment'] = sentiment_label
        if has_seller: result['Seller_Sentiment'] = sentiment_label
        if has_cs: result['CS_Sentiment'] = sentiment_label

    # ---------------------------------------------------------
    # Bước 2: NÂNG CẤP LỚP GHI ĐÈ NGHIỆP VỤ (OVERRIDES)
    # ---------------------------------------------------------

    # 1. Giao nhầm / Không như mong đợi
    wrong_item_phrases = ['errado', 'diferente', 'trocado', 'não foi o que comprei', 'como imaginei', 'nao era o que esperava']
    if any(phrase in review_lower for phrase in wrong_item_phrases):
        result['Seller_Sentiment'] = 'NEG'
        result['Product_Sentiment'] = 'NEU' # Ép về NEU vì khách chưa xài đồ thật
        result['Logistics_Sentiment'] = 'NEU' # Xóa oan cho Logistics

    # 2. Giao thiếu hàng (RegEx)
    missing_item_phrases = ['faltando', 'faltou', 'veio sem', 'incompleto', 'só recebi']
    partial_delivery_pattern = re.compile(r'recebi.*mas.*não')
    if any(phrase in review_lower for phrase in missing_item_phrases) or partial_delivery_pattern.search(review_lower):
        result['Seller_Sentiment'] = 'NEG'

    # 3. Yêu cầu Trả hàng / Hoàn tiền
    return_phrases = ['devolução', 'devolver', 'dinheiro de volta', 'reembolso', 'estorno']
    if any(phrase in review_lower for phrase in return_phrases):
        result['Product_Sentiment'] = 'NEG'
        result['CS_Sentiment'] = 'NEG'

    # 4. LỖI CHẤT LƯỢNG KHÁCH QUAN
    # Bắt các tính từ/động từ mô tả sự vật bị hỏng hóc, kém chất lượng mà AI bỏ sót
    factual_defects = ['não funciona', 'quebrado', 'estragado', 'rasgado', 'não é', 'molha', 'defeito', 'ruim', 'péssimo', 'fino demais']
    if any(phrase in review_lower for phrase in factual_defects) and has_product:
        result['Product_Sentiment'] = 'NEG'

    return result

In [ ]:
df_reviews = pd.read_csv('/content/drive/MyDrive/Data for Colabs/olist_order_reviews.csv')

In [ ]:
df_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01,2018-03-02


In [ ]:
df_reviews.drop(columns=['review_creation_date','review_answer_timestamp','review_comment_title'], inplace=True)

In [ ]:
df_sample = df_reviews.dropna(subset=['review_comment_message']).sample(20, random_state=42).copy()


print("Đang xử lý phân tích...")
# Tách dict kết quả thành 2 cột riêng biệt trong DataFrame
df_sample[['Product_Sentiment', 'Logistics_Sentiment', 'Seller_Sentiment', 'CS_Sentiment']] = df_sample['review_comment_message'].apply(
    lambda x: pd.Series(analyze_review_aspects_v2(x))
)

# In kết quả ra màn hình để nghiệm thu
pd.set_option('display.max_colwidth', None) # Hiển thị full câu review
print(df_sample[['review_comment_message', 'Product_Sentiment', 'Logistics_Sentiment', 'Seller_Sentiment', 'CS_Sentiment']])

Đang xử lý phân tích...
                                                                                                                                                                                             review_comment_message  \
161537                                                                                                                                                                    O TAPETE NÃO VEIO, SOMENTE O MOSQUITEIRO.   
14937                                        Ainda não recebi o produto,pois comprei 3 e apenas 1 foi entregue e ninguém entrou em contato p dar uma explicação, nem mesmo a loja Américanas,quero meus produtos \n   
163374  COMPREI O PRODUTO EMBLOLO DE DESCARGA VALVULA HYDRA ORIENTE COM O PARCEIRO MARKETIN PLAZE targaryen RECEBI O PRODUTO ERRADO NO LUGAR VEIO CONJUNTO DE MANOLA COM PASTILHAS NO LUGAR DO PRODUTO QUE COMPREI.   
57352                                                                                                               